# 🧠 UTS Pembelajaran Mesin — OASIS-1 Alzheimer Classification
## End-to-End Machine Learning Pipeline

| Item | Detail |
|------|--------|
| **Dataset** | OASIS-1 Cross-Sectional — Washington University + Harvard + NIH |
| **Sumber** | https://sites.wustl.edu/oasisbrains/home/oasis-1/ |
| **Referensi** | Marcus et al. (2007), J. Cognitive Neuroscience, 19(9) |
| **Mata Kuliah** | Pembelajaran Mesin |
| **Dosen** | Junta Zeniarja, M.Kom |

---
## Struktur Notebook
- **SOAL 01** — Problem Framing & Konsep ML
- **SOAL 02** — Audit Data & Preprocessing
- **SOAL 03** — Seleksi Fitur, Split, Desain Eksperimen
- **SOAL 04** — Implementasi & Perbandingan Algoritma
- **SOAL 05** — Analisis, Rekomendasi & Demo Inference

---
# 📦 SETUP & IMPORT

In [ ]:
# ============================================================
# GLOBAL SETUP — Jalankan sel ini pertama kali
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import time, warnings, json, os
warnings.filterwarnings('ignore')

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('='*65)
print('  UTS Pembelajaran Mesin — OASIS-1 Alzheimer Classification')
print('  Dataset: Washington University + Harvard + NIH')
print('='*65)
print(f'  Random Seed : {RANDOM_SEED}')
print(f'  NumPy       : {np.__version__}')
print(f'  Pandas      : {pd.__version__}')

---
# 📖 SOAL 01 — Konsep Dasar ML & Problem Framing
**Bobot:** 20% | **Sub-CPMK:** A1164405-81.1

## 1. Pengertian Pembelajaran Mesin & Paradigma

**Pembelajaran mesin** memungkinkan komputer belajar dari data tanpa diprogram eksplisit (Mitchell, 1997).

| Paradigma | Label | Contoh Kasus |
|-----------|-------|-------------|
| **Supervised** | ✅ Ada | Klasifikasi Demented/Nondemented |
| **Unsupervised** | ❌ Tidak | Segmentasi pola atrofi otak |
| **Semi-Supervised** | ⚠️ Sebagian | Data wearable tanpa label PSG |

## 2. Mengapa Ini Klasifikasi?
- Target `Group` bersifat **diskrit kategoris** (Demented / Nondemented)
- **Bukan regresi**: tidak ada nilai kontinu yang diprediksi
- **Bukan klasterisasi**: label sudah tersedia dari dokter spesialis
- Keputusan klinis bersifat **biner**: demensia atau tidak?

In [ ]:
# ============================================================
# SOAL 01 — LOAD DATASET & IDENTIFIKASI ATRIBUT
# ============================================================

# Coba load dari file resmi OASIS, fallback ke simulasi
try:
    df = pd.read_csv('oasis_cross-sectional.csv')
    print('✅ Dataset OASIS-1 resmi dimuat')
except FileNotFoundError:
    # Simulasi dataset identik struktur OASIS-1
    n = 436
    groups = np.random.choice(['Nondemented','Demented'], n, p=[0.54,0.46])
    cdr_v  = [0.0 if g=='Nondemented' else np.random.choice([0.5,1.0,2.0],p=[0.6,0.3,0.1]) for g in groups]
    cdr    = np.array(cdr_v)
    mmse   = np.where(cdr==0,np.random.randint(27,31,n),
             np.where(cdr==0.5,np.random.randint(22,29,n),
             np.where(cdr==1.0,np.random.randint(15,25,n),np.random.randint(5,18,n))))
    age    = np.random.randint(60,97,n)
    nwbv   = (0.82-0.002*age+np.where(cdr>0,-0.03,0)+np.random.normal(0,0.01,n)).round(3)
    etiv   = np.random.randint(1100,1900,n)
    ses    = np.random.choice([1.0,2.0,3.0,4.0,5.0,np.nan],n,p=[0.15,0.25,0.30,0.20,0.05,0.05])
    df = pd.DataFrame({
        'Subject_ID':[f'OAS1_{i:04d}_MR1' for i in range(1,n+1)],
        'MF':np.random.choice(['M','F'],n,p=[0.42,0.58]),
        'Hand':np.random.choice(['R','L'],n,p=[0.95,0.05]),
        'Age':age, 'EDUC':np.random.choice([1,2,3,4,5],n,p=[0.05,0.15,0.40,0.25,0.15]),
        'SES':ses, 'MMSE':mmse, 'CDR':cdr,
        'eTIV':etiv, 'nWBV':nwbv, 'ASF':(1.0/(etiv/1500)).round(3),
        'Group':groups
    })
    print('✅ Dataset OASIS-1 (simulasi identik) dimuat')

df_raw = df.copy()
print(f'Dimensi : {df.shape[0]} baris × {df.shape[1]} kolom')
print(f'Target  : Group — {df["Group"].value_counts().to_dict()}')
df.head()

In [ ]:
# Visualisasi distribusi kelas
fig, axes = plt.subplots(1,2,figsize=(14,5))
fig.patch.set_facecolor('#F4F6F9')

# Distribusi Group
ax = axes[0]; ax.set_facecolor('white')
vc = df['Group'].value_counts()
bars = ax.bar(vc.index, vc.values, color=['#2ECC71','#E74C3C'], edgecolor='white', linewidth=2, width=0.55)
for bar, val in zip(bars, vc.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            f'{val} ({val/len(df)*100:.1f}%)', ha='center', fontsize=12, fontweight='bold')
ax.set_title('Distribusi Kelas Target (Group)', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# Distribusi CDR
ax = axes[1]; ax.set_facecolor('white')
cdr_c = df['CDR'].value_counts().sort_index()
bars = ax.bar([f'CDR {k}' for k in cdr_c.index], cdr_c.values,
              color=['#2ECC71','#F39C12','#E74C3C','#8E44AD'], edgecolor='white', linewidth=2)
for bar, val in zip(bars, cdr_c.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2,
            str(val), ha='center', fontsize=12, fontweight='bold')
ax.set_title('Distribusi CDR (Clinical Dementia Rating)', fontsize=13, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.show()
print('\nPotensi tantangan:')
print(f'  Class imbalance : {vc.max()/vc.min():.2f}:1')
print(f'  Missing SES     : {df["SES"].isnull().sum()} baris')
print(f'  Fitur kategorikal: MF, Hand → perlu encoding')
print(f'  Multikolinear   : ASF ↔ eTIV (r≈-0.99) → drop ASF')

---
# 🔍 SOAL 02 — Audit Dataset & Preprocessing
**Bobot:** 20% | **Sub-CPMK:** A1164405-101.1

In [ ]:
# ============================================================
# SOAL 02 — AUDIT & PIPELINE PREPROCESSING
# ============================================================
from sklearn.preprocessing import StandardScaler, LabelEncoder

print('AUDIT DATA LENGKAP'); print('='*55)
print(f'Dimensi         : {df.shape}')
print(f'Missing values  :\n{df.isnull().sum()[df.isnull().sum()>0]}')
print(f'Duplikat        : {df.duplicated().sum()}')

# ── PIPELINE ────────────────────────────────────────────────
df_clean = df.copy()

# 1. Drop ID & Hand
df_clean.drop(columns=['Subject_ID','Hand'], inplace=True, errors='ignore')
print('\n[1] Drop Subject_ID & Hand ✅')

# 2. Imputasi SES
df_clean['SES'] = df_clean.groupby('Group')['SES'].transform(lambda x: x.fillna(x.median()))
print(f'[2] Imputasi SES (median/Group): {df_clean["SES"].isnull().sum()} missing ✅')

# 3. Encoding
df_clean['MF'] = (df_clean['MF']=='F').astype(int)
df_clean['BMI_fix'] = df_clean.get('BMI Category','').map({'Normal':0,'Overweight':1,'Obese':2}) if 'BMI Category' in df_clean.columns else 0
df_clean['Group_Encoded'] = (df_clean['Group']=='Demented').astype(int)
print('[3] Encoding MF & Target ✅')

# 4. Drop ASF (multikolinear)
df_clean.drop(columns=['ASF'], inplace=True, errors='ignore')
print('[4] Drop ASF (multikolinear eTIV r≈-0.99) ✅')

# 5. Outlier check
FEAT_NUM = ['Age','EDUC','SES','MMSE','CDR','eTIV','nWBV']
print('\n[5] Outlier IQR check:')
for col in FEAT_NUM:
    Q1,Q3 = df_clean[col].quantile(0.25), df_clean[col].quantile(0.75)
    IQR   = Q3-Q1
    n_out = ((df_clean[col]<Q1-1.5*IQR)|(df_clean[col]>Q3+1.5*IQR)).sum()
    print(f'   {col:<8}: {n_out} outlier → {"dipertahankan (valid klinis)" if n_out>0 else "bersih ✅"}')

# 6. StandardScaler
FEAT_FINAL = ['MF','Age','EDUC','SES','MMSE','CDR','eTIV','nWBV']
FEAT_FINAL = [f for f in FEAT_FINAL if f in df_clean.columns]
scaler     = StandardScaler()
df_scaled  = df_clean.copy()
df_scaled[FEAT_FINAL] = scaler.fit_transform(df_clean[FEAT_FINAL])
print(f'\n[6] StandardScaler → mean≈0, std≈1 ✅')
print(f'    Dimensi final: {df_scaled.shape}')

In [ ]:
# Before vs After Visualisasi
fig, axes = plt.subplots(2,4,figsize=(18,9))
fig.patch.set_facecolor('#F4F6F9')
fig.suptitle('Distribusi Fitur: BEFORE (atas) vs AFTER StandardScaler (bawah)',
             fontsize=14, fontweight='bold')
cols   = ['Age','MMSE','CDR','nWBV']
colors = ['#3498DB','#E74C3C','#2ECC71','#F39C12']

for i,(col,color) in enumerate(zip(cols,colors)):
    # Before
    axes[0,i].set_facecolor('white')
    axes[0,i].hist(df_clean[col], bins=20, color=color, alpha=0.85, edgecolor='white')
    axes[0,i].set_title(f'{col} (Before)', fontsize=11, fontweight='bold')
    axes[0,i].spines['top'].set_visible(False); axes[0,i].spines['right'].set_visible(False)
    # After
    axes[1,i].set_facecolor('white')
    axes[1,i].hist(df_scaled[col], bins=20, color=color, alpha=0.85, edgecolor='white')
    axes[1,i].axvline(0, color='red', linestyle='--', linewidth=1.8)
    axes[1,i].set_title(f'{col} scaled (After)', fontsize=11, fontweight='bold')
    axes[1,i].spines['top'].set_visible(False); axes[1,i].spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
print('✅ Before/After preprocessing divisualisasikan')

---
# 🎯 SOAL 03 — Seleksi Fitur, Split Data & Desain Eksperimen
**Bobot:** 20% | **Sub-CPMK:** A1164405-101.1

In [ ]:
# ============================================================
# SOAL 03 — FEATURE SELECTION & STRATIFIED SPLIT
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.metrics import accuracy_score

TARGET = 'Group_Encoded'
FEAT_FINAL = [f for f in ['MF','Age','EDUC','SES','MMSE','CDR','eTIV','nWBV'] if f in df_scaled.columns]

X = df_scaled[FEAT_FINAL].values
y = df_scaled[TARGET].values

# Feature importance
f_scores,f_pvals = f_classif(X,y)
mi_scores        = mutual_info_classif(X,y,random_state=RANDOM_SEED)

fi_df = pd.DataFrame({'Fitur':FEAT_FINAL,'F-Score':f_scores.round(2),
                       'p-value':f_pvals.round(4),'MI':mi_scores.round(4)
                      }).sort_values('F-Score',ascending=False)
print('Feature Importance Ranking:')
print(fi_df.to_string(index=False))

# Stratified Split 80:20
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=RANDOM_SEED,stratify=y)
print(f'\n✅ Stratified Split: Train={len(X_train)} | Test={len(X_test)}')

# Baseline
dummy = DummyClassifier(strategy='most_frequent',random_state=RANDOM_SEED)
dummy.fit(X_train,y_train)
baseline = accuracy_score(y_test,dummy.predict(X_test))
print(f'📏 Baseline accuracy: {baseline:.4f} ({baseline*100:.2f}%)')
print(f'   → Semua model wajib melampaui {baseline*100:.2f}%')

In [ ]:
# Visualisasi split
fig,axes = plt.subplots(1,3,figsize=(14,5))
fig.patch.set_facecolor('#F4F6F9')
fig.suptitle('Stratified Split 80:20 — OASIS-1',fontsize=13,fontweight='bold')
for ax,(title,data,n,color) in zip(axes,[
    ('Full Dataset',y,len(y),'#1A5276'),
    ('Train (80%)',y_train,len(y_train),'#2471A3'),
    ('Test (20%)',y_test,len(y_test),'#8E44AD')]):
    ax.set_facecolor('white')
    counts=[np.sum(data==0),np.sum(data==1)]
    bars=ax.bar(['Nondemented','Demented'],counts,
                color=['#2ECC71','#E74C3C'],edgecolor='white',linewidth=2,width=0.55)
    for bar,val in zip(bars,counts):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+1,
                f'{val}\n({val/n*100:.1f}%)',ha='center',fontsize=11,fontweight='bold')
    ax.set_title(f'{title} (n={n})',fontsize=11,fontweight='bold',color=color)
    ax.set_ylim(0,max(counts)*1.3)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

---
# 📊 SOAL 04 — Implementasi & Perbandingan Algoritma
**Bobot:** 20% | **Sub-CPMK:** A1164405-81.2

In [ ]:
# ============================================================
# SOAL 04 — TRAIN & EVALUATE 3 ALGORITMA
# ============================================================
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (precision_score,recall_score,f1_score,
                              confusion_matrix,classification_report)

LABELS = ['Nondemented','Demented']
models = {
    'KNN (k=5)':      KNeighborsClassifier(n_neighbors=5,metric='euclidean'),
    'Decision Tree':  DecisionTreeClassifier(criterion='gini',random_state=RANDOM_SEED,class_weight='balanced'),
    'Naive Bayes':    GaussianNB(var_smoothing=1e-9),
}

results = {}
for name,model in models.items():
    t0=time.time(); model.fit(X_train,y_train); tr_ms=(time.time()-t0)*1000
    t0=time.time(); y_pred=model.predict(X_test); pr_ms=(time.time()-t0)*1000
    results[name] = {
        'model':model,'y_pred':y_pred,'cm':confusion_matrix(y_test,y_pred),
        'accuracy' :accuracy_score(y_test,y_pred),
        'precision':precision_score(y_test,y_pred,average='weighted',zero_division=0),
        'recall'   :recall_score(y_test,y_pred,average='weighted',zero_division=0),
        'f1_score' :f1_score(y_test,y_pred,average='weighted',zero_division=0),
        'train_ms':tr_ms,'pred_ms':pr_ms
    }
    print(f'\n✅ {name}')
    print(classification_report(y_test,y_pred,target_names=LABELS,zero_division=0))

In [ ]:
# Confusion Matrix & Perbandingan Visual
fig,axes = plt.subplots(1,3,figsize=(16,5))
fig.patch.set_facecolor('#F4F6F9')
fig.suptitle('Confusion Matrix — Semua Model | OASIS-1',fontsize=14,fontweight='bold')

for ax,(name,res),cmap,color in zip(axes,results.items(),
    ['Blues','Oranges','Reds'],['#2471A3','#BA4A00','#922B21']):
    sns.heatmap(res['cm'],annot=True,fmt='d',cmap=cmap,ax=ax,
                xticklabels=LABELS,yticklabels=LABELS,cbar=False,
                linewidths=0.5,annot_kws={'size':14,'fontweight':'bold'})
    ax.set_title(f'{name}\nAcc={res["accuracy"]:.3f} | F1={res["f1_score"]:.3f}',
                 fontsize=11,fontweight='bold',color=color)
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

# Tabel ringkasan
print('\nTABEL KOMPARATIF'); print('='*65)
print(f'{"Model":<18} {"Accuracy":>10} {"Precision":>10} {"Recall":>10} {"F1-Score":>10}')
print('-'*65)
for name,res in results.items():
    print(f'{name:<18} {res["accuracy"]:>10.4f} {res["precision"]:>10.4f} {res["recall"]:>10.4f} {res["f1_score"]:>10.4f}')

In [ ]:
# Heatmap komparatif
fig,ax = plt.subplots(figsize=(10,4))
fig.patch.set_facecolor('#F4F6F9'); ax.set_facecolor('white')
comp = pd.DataFrame({n:{'Accuracy':r['accuracy'],'Precision':r['precision'],
                         'Recall':r['recall'],'F1-Score':r['f1_score']}
                     for n,r in results.items()}).T
sns.heatmap(comp,annot=True,fmt='.4f',cmap='YlOrRd',ax=ax,
            linewidths=0.5,annot_kws={'size':13,'fontweight':'bold'})
ax.set_title('Heatmap Komparatif — Semua Model & Metrik',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.show()

---
# 🏆 SOAL 05 — Analisis, Rekomendasi & Demo Inference
**Bobot:** 20% | **Sub-CPMK:** A1164405-81.2

In [ ]:
# ============================================================
# SOAL 05 — ANALISIS, REKOMENDASI & ERROR ANALYSIS
# ============================================================
import joblib

# Tentukan model terbaik
best_name = max(results, key=lambda n: results[n]['f1_score'])
best_model= results[best_name]['model']
print(f'🏆 Model Terbaik   : {best_name}')
print(f'   Accuracy        : {results[best_name]["accuracy"]:.4f}')
print(f'   F1-Score        : {results[best_name]["f1_score"]:.4f}')

# Analisis komparatif
print('\nANALISIS KOMPARATIF')
print('='*65)
analysis = {
    'KNN (k=5)'      :{'Interpretabilitas':'Rendah','Kompleksitas':'O(n×d) prediksi',
                       'Kestabilan':'Tinggi','Skenario ideal':'Data baru kontinu'},
    'Decision Tree'  :{'Interpretabilitas':'Sangat Tinggi','Kompleksitas':'O(log n) prediksi',
                       'Kestabilan':'Rendah (overfitting risk)','Skenario ideal':'Butuh interpretasi klinis'},
    'Naive Bayes'    :{'Interpretabilitas':'Menengah','Kompleksitas':'O(d) prediksi',
                       'Kestabilan':'Tinggi','Skenario ideal':'Resource terbatas'},
}
for model,info in analysis.items():
    print(f'\n{model}:')
    for k,v in info.items(): print(f'  {k:<20}: {v}')

In [ ]:
# Error Analysis
print('ERROR ANALYSIS — KNN (model paling reliable)'); print('='*55)
knn_pred = results['KNN (k=5)']['y_pred']
df_test  = pd.DataFrame(X_test, columns=FEAT_FINAL)
df_test['Actual']    = ['Demented' if y==1 else 'Nondemented' for y in y_test]
df_test['Predicted'] = ['Demented' if y==1 else 'Nondemented' for y in knn_pred]
df_test['Correct']   = y_test == knn_pred

errors = df_test[~df_test['Correct']]
print(f'Total error KNN: {len(errors)} dari {len(y_test)} test samples ({len(errors)/len(y_test)*100:.1f}%)')
print(f'\nPola misclassification:')
for _, row in errors.iterrows():
    print(f'  Actual={row["Actual"]} → Predicted={row["Predicted"]} | MMSE={row["MMSE"]:.2f} CDR={row["CDR"]:.2f}')

In [ ]:
# Simpan best model
joblib.dump(best_model, 'best_model.joblib')
joblib.dump(scaler,     'scaler.joblib')
print(f'✅ Model terbaik ({best_name}) tersimpan: best_model.joblib')
print(f'✅ Scaler tersimpan: scaler.joblib')

In [ ]:
# ============================================================
# DEMO INFERENCE — Prediksi Baru
# ============================================================
print('DEMO PREDIKSI — Contoh Kasus Baru'); print('='*55)

# Contoh pasien baru
test_cases = [
    {'name':'Pasien A (Berisiko Rendah)', 'data':[0,68,4,2,28,0.0,1500,0.76]},
    {'name':'Pasien B (Berisiko Tinggi)', 'data':[1,79,2,4,18,1.0,1200,0.65]},
    {'name':'Pasien C (Borderline)',      'data':[0,72,3,3,24,0.5,1450,0.71]},
]

for tc in test_cases:
    X_new = scaler.transform([tc['data']])
    pred  = best_model.predict(X_new)[0]
    label = 'DEMENTED 🔴' if pred==1 else 'NONDEMENTED 🟢'
    if hasattr(best_model,'predict_proba'):
        prob = best_model.predict_proba(X_new)[0]
        conf = f'(conf: {max(prob)*100:.1f}%)'
    else:
        conf = ''
    print(f'\n  {tc["name"]}')
    print(f'  Input : MF={tc["data"][0]}, Age={tc["data"][1]}, EDUC={tc["data"][2]}')
    print(f'          MMSE={tc["data"][4]}, CDR={tc["data"][5]}, nWBV={tc["data"][7]}')
    print(f'  Hasil : {label} {conf}')

print('\n✅ SEMUA SOAL UTS SELESAI!')
print('='*55)
print('Referensi:')
print('  [1] Marcus et al. (2007). OASIS. J. Cognitive Neuroscience, 19(9).')
print('  [2] Cabanillas-Carbonell (2025). Brain Behav Immun Integr. Q1 Scopus.')
print('  [3] Lazli (2025). JMIRx Med, 6:e60866. PubMed/PMC.')
print('  [4] Sarica et al. (2023). Front. Aging Neuroscience. Q1 Scopus.')